In [ ]:
import mpramnist
from mpramnist.Agarwal2025.dataset import AgarwalSingleDataset

# MPRA
from mpramnist.Siraj2026.dataset import SirajMPRADataset

# SatMut
from mpramnist.Siraj2026.dataset import SirajSatMutDataset

from mpramnist.Siraj2026.trainer import LitModel_Siraj

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights
import mpramnist.transforms as t

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import lightning.pytorch as L
from lightning.pytorch.callbacks import ModelCheckpoint

from torchmetrics import PearsonCorrCoef

BATCH_SIZE = 1024
NUM_WORKERS = 8


# Siraj Datasets and Experimental Design

The Siraj MPRA dataset is based on Massively Parallel Reporter Assay (MPRA) functional screens designed to systematically evaluate common variants underlying complex and molecular human traits. The study characterized 221,412 statistically fine-mapped, trait-associated genetic variants along with corresponding control sequences. To capture tissue-specific regulatory architectures, these variants were assayed across 5 diverse human cell types, leading to the identification of 13,121 high-confidence expression-modulating regulatory variants (emVars) with high precision. Mechanistic annotation of these functional perturbations revealed that only 69% could be explained by the disruption of a known, canonical transcription factor (TF) binding motif, highlighting a substantial dependence on non-canonical sequence context.

The Siraj SatMut dataset represents a high-density, single-nucleotide resolution functional dissection of localized regulatory grammar using saturation mutagenesis. The experiment targeted 136 functional regulatory variants identified in the primary MPRA screen that lacked an obvious canonical mechanism. By systematically introducing every possible single-nucleotide substitution across the contiguous cis-regulatory windows flanking these 136 loci, the assay mapped complete local sequence-to-function landscapes. This dense mutational scanning resolved the exact biophysical mechanisms of the elements, successfully assigning affected transcription factors to 91% of the non-canonical variants. Furthermore, the dataset directly quantifies regulatory epistasis, demonstrating that 11% of the tested regulatory variants in close physical proximity exhibit non-linear, epistatic sequence interactions.

# Proposed SirajDatasets Benchmark Application

These experimentally validated sequences are recommended as a reference dataset for evaluating the performance of machine learning models.

# Current Workflow

In this notebook, we:

1. Train the **MPRALegNet** model on the **AgarwalSingle dataset**

2. Assess its predictive power using **Siraj's MPRA data or SatMut data**

Focus: Sequences tested in **HepG2** cell line

# Pretrain on Agarwal's HepG2

In [ ]:
# shift (0,15)
# preprocessing
shift = 15

train_transform = t.Compose([
    t.AddFlanks(
        AgarwalSingleDataset.CONSTANT_LEFT_FLANK,
        AgarwalSingleDataset.CONSTANT_RIGHT_FLANK,
    ),
    t.AddFlanks("", AgarwalSingleDataset.RIGHT_FLANK),
    t.RightCrop(230, 260),
    t.LeftCrop(230, 230),
    t.ReverseComplement(0.5),
    t.Seq2Tensor(),
])
test_transform = t.Compose([
    t.AddFlanks(
        AgarwalSingleDataset.CONSTANT_LEFT_FLANK,
        AgarwalSingleDataset.CONSTANT_RIGHT_FLANK,
    ),
    t.ReverseComplement(0),
    t.Seq2Tensor(),
])

# load the data
Cell_Type = "HepG2"  # or K562
train_dataset = AgarwalSingleDataset(
    cell_type=Cell_Type,
    split="train",
    transform=train_transform,
    root="../data/",
)
val_dataset = AgarwalSingleDataset(
    cell_type=Cell_Type,
    split="val",
    transform=test_transform,
    root="../data/",
)
test_dataset = AgarwalSingleDataset(
    cell_type=Cell_Type,
    split="test",
    transform=test_transform,
    root="../data/",
)

# encapsulate data into dataloader form
train_loader = DataLoader(
    dataset=train_dataset, batch_size=1024, shuffle=True, num_workers=103
)
val_loader = DataLoader(
    dataset=val_dataset, batch_size=1024, shuffle=False, num_workers=103
)
test_loader = DataLoader(
    dataset=test_dataset, batch_size=1024, shuffle=False, num_workers=103
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

in_channels = len(train_dataset[0][0])
out_channels = 1


98336 12292 12298


In [ ]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Siraj(
    model=model, loss=nn.MSELoss(), weight_decay=1e-1, lr=1e-2, print_each=10
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_pearson", mode="max", save_top_k=1, save_last=False
)
# Initialize a trainer
trainer_hepg2 = L.Trainer(
    accelerator="gpu",
    devices=[1],
    max_epochs=5,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
    callbacks=[checkpoint_callback],
)

trainer_hepg2.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.

Epoch 4: 100%|██████████| 97/97 [00:17<00:00,  5.46it/s, v_num=3, val_loss=0.313, val_pearson=0.697, train_loss=0.277]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 97/97 [00:17<00:00,  5.43it/s, v_num=3, val_loss=0.313, val_pearson=0.697, train_loss=0.277]


In [ ]:
best_model_path = checkpoint_callback.best_model_path
seq_model_hepg2 = LitModel_Siraj.load_from_checkpoint(
    best_model_path,
    model=model,
    loss=nn.MSELoss(),
    weight_decay=0.1,
    lr=0.01,
    print_each=1,
)


## Siraj's MPRA sequences evaluation

In [ ]:
print(f"Avalable cell lines for MPRA dataset: {SirajMPRADataset.CELL_TYPE}")


In [ ]:
def meaned_prediction(forw, rev, trainer, seq_model, name, is_kircher=False):
    predictions_forw = trainer.predict(seq_model, dataloaders=forw)
    targets = torch.cat([pred["target"] for pred in predictions_forw])
    y_preds_forw = torch.cat([pred["ref_predicted"] for pred in predictions_forw])

    predictions_rev = trainer.predict(seq_model, dataloaders=rev)
    y_preds_rev = torch.cat([pred["ref_predicted"] for pred in predictions_rev])

    mean_forw = torch.mean(torch.stack([y_preds_forw, y_preds_rev]), dim=0)

    pears = PearsonCorrCoef()
    print(name, " Pearson correlation")

    if is_kircher:
        y_preds_forw_alt = torch.cat([
            pred["alt_predicted"] for pred in predictions_forw
        ])
        y_preds_rev_alt = torch.cat([pred["alt_predicted"] for pred in predictions_rev])
        mean_alt = torch.mean(torch.stack([y_preds_forw_alt, y_preds_rev_alt]), dim=0)
        pred = mean_alt - mean_forw
        return pears(pred, targets)

    return pears(mean_forw, targets)


In [ ]:
forw_transform = t.Compose([
    t.AddFlanks(
        SirajMPRADataset.CONSTANT_LEFT_FLANK, SirajMPRADataset.CONSTANT_RIGHT_FLANK
    ),
    t.Seq2Tensor(),
])
rev_transform = t.Compose([
    t.AddFlanks(
        SirajMPRADataset.CONSTANT_LEFT_FLANK, SirajMPRADataset.CONSTANT_RIGHT_FLANK
    ),
    t.ReverseComplement(1),
    t.Seq2Tensor(),
])

hepg2_mpra_dataset_forw = SirajMPRADataset(
    length=200, cell_type="HEPG2", transform=forw_transform, root="../data/"
)
hepg2_mpra_dataset_rev = SirajMPRADataset(
    length=200, cell_type="HEPG2", transform=rev_transform, root="../data/"
)

print("MPRA dataset info for HEPG2")
print(len(hepg2_mpra_dataset_forw))

hepg2_mpra_forw = DataLoader(
    dataset=hepg2_mpra_dataset_forw,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
hepg2_mpra_rev = DataLoader(
    dataset=hepg2_mpra_dataset_rev,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("==========")
pearson = meaned_prediction(
    hepg2_mpra_forw,
    hepg2_mpra_rev,
    trainer_hepg2,
    seq_model_hepg2,
    name="HEPG2",
    is_kircher=True,
)
print(pearson)
print("==========")


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


LDLR info
2176
Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 29.26it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 24.49it/s]
LDLR  Pearson correlation
tensor(0.5239)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


...


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 31.42it/s]
F9  Pearson correlation
tensor(0.5321)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


SORT1 info
5898
Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 27.09it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 35.58it/s]
F9  Pearson correlation
tensor(0.3764)


## Siraj's SatMut sequences evaluation

In [ ]:
print(f"Avalable cell lines for SatMut dataset: {SirajSatMutDataset.CELL_TYPE}")


In [ ]:
def meaned_prediction(forw, rev, trainer, seq_model, name, is_kircher=False):
    predictions_forw = trainer.predict(seq_model, dataloaders=forw)
    targets = torch.cat([pred["target"] for pred in predictions_forw])
    y_preds_forw = torch.cat([pred["ref_predicted"] for pred in predictions_forw])

    predictions_rev = trainer.predict(seq_model, dataloaders=rev)
    y_preds_rev = torch.cat([pred["ref_predicted"] for pred in predictions_rev])

    mean_forw = torch.mean(torch.stack([y_preds_forw, y_preds_rev]), dim=0)

    pears = PearsonCorrCoef()
    print(name, " Pearson correlation")

    if is_kircher:
        y_preds_forw_alt = torch.cat([
            pred["alt_predicted"] for pred in predictions_forw
        ])
        y_preds_rev_alt = torch.cat([pred["alt_predicted"] for pred in predictions_rev])
        mean_alt = torch.mean(torch.stack([y_preds_forw_alt, y_preds_rev_alt]), dim=0)
        pred = mean_alt - mean_forw
        return pears(pred, targets)

    return pears(mean_forw, targets)


In [ ]:
forw_transform = t.Compose([
    t.AddFlanks(
        SirajSatMutDataset.CONSTANT_LEFT_FLANK, SirajSatMutDataset.CONSTANT_RIGHT_FLANK
    ),
    t.Seq2Tensor(),
])
rev_transform = t.Compose([
    t.AddFlanks(
        SirajSatMutDataset.CONSTANT_LEFT_FLANK, SirajSatMutDataset.CONSTANT_RIGHT_FLANK
    ),
    t.ReverseComplement(1),
    t.Seq2Tensor(),
])


In [ ]:
mut1_hepg2_satmut_dataset_forw = SirajSatMutDataset(
    length=200, cell_type="HEPG2", mut_num=1, transform=forw_transform, root="../data/"
)
mut1_hepg2_satmut_dataset_rev = SirajSatMutDataset(
    length=200, cell_type="HEPG2", mut_num=1, transform=rev_transform, root="../data/"
)

print("HEPG2 info for sequences with 1 baseline mutation")
print(len(mut1_hepg2_satmut_dataset_forw))

mut1_hepg2_satmut_forw = DataLoader(
    dataset=mut1_hepg2_satmut_dataset_forw,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
mut1_hepg2_satmut_rev = DataLoader(
    dataset=mut1_hepg2_satmut_dataset_rev,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("==========")
pearson = meaned_prediction(
    mut1_hepg2_satmut_forw,
    mut1_hepg2_satmut_rev,
    trainer_hepg2,
    seq_model_hepg2,
    name="HEPG2",
    is_kircher=True,
)
print(pearson)
print("==========")


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


LDLR info
2176
Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 29.26it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 24.49it/s]
LDLR  Pearson correlation
tensor(0.5239)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


...


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 31.42it/s]
F9  Pearson correlation
tensor(0.5321)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


SORT1 info
5898
Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 27.09it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 35.58it/s]
F9  Pearson correlation
tensor(0.3764)


In [ ]:
mut2_hepg2_satmut_dataset_forw = SirajSatMutDataset(
    length=200, cell_type="HEPG2", mut_num=2, transform=forw_transform, root="../data/"
)
mut2_hepg2_satmut_dataset_rev = SirajSatMutDataset(
    length=200, cell_type="HEPG2", mut_num=2, transform=rev_transform, root="../data/"
)

print("HEPG2 info for sequences with 2 baseline mutations")
print(len(mut2_hepg2_satmut_dataset_forw))

mut2_hepg2_satmut_forw = DataLoader(
    dataset=mut2_hepg2_satmut_dataset_forw,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
mut2_hepg2_satmut_rev = DataLoader(
    dataset=mut2_hepg2_satmut_dataset_rev,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("==========")
pearson = meaned_prediction(
    mut2_hepg2_satmut_forw,
    mut2_hepg2_satmut_rev,
    trainer_hepg2,
    seq_model_hepg2,
    name="HEPG2",
    is_kircher=True,
)
print(pearson)
print("==========")
